# 03: Smoke Test + Baselines + ORA

Workhorse notebook. Runs `grounded-vla smoke`, then evaluates the three agents
(ReAct + Mistral, single-shot LLaVA, ORA + LLaVA) across ScienceQA, the
synthetic sample, and Mind2Web. All artifacts go to
`/content/drive/MyDrive/grounded_vla/runs/<agent>__<dataset>/`.

**Runtime:** A100 GPU **strongly recommended** (Pro+ default).

**Estimated wallclock:** ~60-90 min on A100 for the full sweep.
**Estimated compute units:** ~15-20 (well within Pro+'s 500/month).

## Pro+ tip: background execution

Tools → Settings → check **'Run after disconnecting'**. With this on, you can
close the tab; Colab keeps the runtime alive for up to 24 hours and the
checkpointed `EvalRunner` will keep writing per-task results to Drive. Re-open
the notebook later and the results will be waiting in Drive.

If a session does die mid-sweep, just re-run — `resume=True` skips already-
completed tasks via the trajectory JSONs already on Drive.

In [ ]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

In [ ]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

In [ ]:
# Clone (first run) or pull (re-run / re-attach) the repo into ephemeral /content.
# This ensures every session picks up the latest fixes from GitHub.
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
else:
    subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
%cd /content/grounded_vla

In [ ]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

In [ ]:
# Point HF cache at the Drive-mounted weights so transformers loads from there.
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
assert os.path.isdir(WEIGHTS), 'run notebook 01 first'
assert os.path.isdir(DATA),    'run notebook 02 first'

In [ ]:
# Smoke test: pure-mock pipeline. Should print 'smoke ok' in <5 seconds.
!grounded-vla smoke

In [ ]:
# Build the three agents. On A100 we can use the default GenerationConfig
# without any concessions to memory.
from grounded_vla.agents import ORAAgent, ReActAgent, SingleShotVLMAgent
from grounded_vla.backends import make_backend
from grounded_vla.backends.base import GenerationConfig

WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'

def build_react():
    backend = make_backend({
        'kind': 'mistral',
        'model_id': f'{WEIGHTS}/Mistral-7B-Instruct-v0.2',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ReActAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

def build_single_shot():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return SingleShotVLMAgent(backend, GenerationConfig(max_new_tokens=256, temperature=0.0))

def build_ora():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ORAAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

In [ ]:
# Build the three datasets pointing at the Drive-mounted data.
from grounded_vla.data import make_dataset
DATA = '/content/drive/MyDrive/grounded_vla/data'

def scienceqa(limit=50):
    return make_dataset({
        'kind': 'scienceqa',
        'jsonl_path': f'{DATA}/scienceqa/test.jsonl',
        'images_dir': f'{DATA}/scienceqa',
        'limit': limit,
    })

def _mind2web_jsonl():
    # Auto-discover whichever split notebook 02 produced (test_task,
    # test_website, train fallback, ...).
    import glob
    candidates = sorted(glob.glob(f'{DATA}/mind2web/*.jsonl'))
    if not candidates:
        raise FileNotFoundError(f'no Mind2Web JSONL under {DATA}/mind2web/')
    return candidates[0]

def mind2web(limit=30):
    return make_dataset({
        'kind': 'mind2web',
        'jsonl_path': _mind2web_jsonl(),
        'images_dir': f'{DATA}/mind2web/images',
        'limit': limit,
    })

def synthetic():
    return make_dataset({
        'kind': 'jsonl',
        'path': f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
        'source': 'synthetic',
    })

In [ ]:
# Generic eval runner with checkpointing -> Drive. Resumable across sessions.
from grounded_vla.eval import EvalRunner
from pathlib import Path

RUNS = Path('/content/drive/MyDrive/grounded_vla/runs')
RUNS.mkdir(parents=True, exist_ok=True)

def run(agent_name, agent, ds_name, dataset, **kw):
    out = RUNS / f'{agent_name}__{ds_name}'
    runner = EvalRunner(agent)
    res = runner.evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True, **kw
    )
    print(f'{agent_name} on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

## ReAct + Mistral (Baseline 1)

Text-only. ~1-2 sec/task on A100 with 4-bit Mistral. ~3 minutes total.

In [ ]:
react = build_react()
_ = run('react_mistral', react, 'scienceqa',  scienceqa(limit=50))
_ = run('react_mistral', react, 'synthetic',  synthetic())
_ = run('react_mistral', react, 'mind2web',   mind2web(limit=30))
react.backend.close()

## Single-shot LLaVA (Baseline 2)

VLM, one call per task. Tests H1 (visual grounding helps).

In [ ]:
ss = build_single_shot()
_ = run('single_shot_llava', ss, 'scienceqa',  scienceqa(limit=50))
_ = run('single_shot_llava', ss, 'synthetic',  synthetic())
_ = run('single_shot_llava', ss, 'mind2web',   mind2web(limit=30))
ss.backend.close()

## ORA + LLaVA (Our Method)

VLM with the Observe-Reason-Act loop and per-step visual re-encoding. Tests H2.
This is the single longest cell in the project (~30-50 min on A100).

In [ ]:
ora = build_ora()
_ = run('ora_llava', ora, 'scienceqa',  scienceqa(limit=50))
_ = run('ora_llava', ora, 'synthetic',  synthetic())
_ = run('ora_llava', ora, 'mind2web',   mind2web(limit=30))
ora.backend.close()

In [ ]:
# Aggregate results table for the report.
import json
from pathlib import Path
rows = []
for d in sorted(Path('/content/drive/MyDrive/grounded_vla/runs').iterdir()):
    s = json.loads((d / 'summary.json').read_text())
    rows.append((d.name, s['n_tasks'], s['task_completion_rate'], s['mean_steps'], s['error_breakdown']))
for r in rows:
    print(f'{r[0]:30s}  n={r[1]:3d}  success={r[2]:.3f}  steps={r[3]:.2f}  errors={r[4]}')

## Done

Results live at `/content/drive/MyDrive/grounded_vla/runs/`. The H1 and H2
comparisons drop straight out of the table above:

- **H1 (vision helps):** compare `react_mistral` vs `single_shot_llava` per dataset.
- **H2 (loop helps):** compare `single_shot_llava` vs `ora_llava` per dataset.

Move on to `04_lora_finetune.ipynb` for the H3 stretch goal.